<a href="https://colab.research.google.com/github/Shrinkhal01/hybrid-xai-lung-nodule/blob/phase-1/phase1preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Install required medical imaging, DICOM, and deep learning libraries

In [ ]:
!pip install -q pydicom SimpleITK pylidc torch h5py pandas numpy matplotlib tqdm
print("✅ Libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 85.5 MB/s eta 0:00:00
✅ Libraries installed successfully!


# Cell 2: Connect Google Drive and specify dataset paths

In [ ]:
from google.colab import drive
import os
from pathlib import Path
# Mount Google Drive
drive.mount('/content/drive')
# Paths in your Google Drive
RAW_DATA_DIR = Path('/content/drive/MyDrive/Lung_Nodule_Project/raw_data')
OUTPUT_DIR = Path('/content/drive/MyDrive/Lung_Nodule_Project/processed_patches')
PATCHES_DIR = OUTPUT_DIR / 'patches'
# Create output directories if they don't already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATCHES_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Raw Data Path: {RAW_DATA_DIR}")
print(f"📁 Processed Patches Path: {PATCHES_DIR}")

Mounted at /content/drive
📁 Raw Data Path: /content/drive/MyDrive/Lung_Nodule_Project/raw_data
📁 Processed Patches Path: /content/drive/MyDrive/Lung_Nodule_Project/processed_patches/patches


# Cell 3: Configure ~/.pylidcrc

In [ ]:
config_path = Path.home() / ".pylidcrc"
config_content = f"""[pylidc]
path = {RAW_DATA_DIR.resolve()}
warn = False
"""
config_path.write_text(config_content)
print(f"✅ Configured ~/.pylidcrc pointing to {RAW_DATA_DIR}")
# Import pylidc
import pylidc as pl
print("✅ pylidc imported and ready!")


✅ Configured ~/.pylidcrc pointing to /content/drive/MyDrive/Lung_Nodule_Project/raw_data
✅ pylidc imported and ready!


# Cell 4: Production Medical Imaging Pipeline Functions

In [ ]:
import sys
import numpy as np
import pandas as pd
import SimpleITK as sitk
import pydicom
import torch
from typing import Tuple, List, Dict, Any, Optional
def find_dicom_series_dir(scan_folder: Path) -> Optional[Path]:
    """Find the directory with the primary CT series (largest number of slices)."""
    candidate_dirs = []
    for root, _, files in os.walk(scan_folder):
        dcm_count = sum(1 for f in files if f.lower().endswith(".dcm") or not "." in f)
        if dcm_count > 10:
            candidate_dirs.append((Path(root), dcm_count))
    if not candidate_dirs:
        return None
    candidate_dirs.sort(key=lambda x: x[1], reverse=True)
    return candidate_dirs[0][0]
def load_dicom_volume_sitk(dicom_dir: Path) -> sitk.Image:
    """Load DICOM series into a 3D SimpleITK image with spatial metadata."""
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(dicom_dir))
    if not series_ids:
        file_names = reader.GetGDCMSeriesFileNames(str(dicom_dir))
    else:
        file_names = reader.GetGDCMSeriesFileNames(str(dicom_dir), series_ids[0])

    if not file_names:
        raise FileNotFoundError(f"No DICOM files found in: {dicom_dir}")

    reader.SetFileNames(file_names)
    return reader.Execute()
def resample_volume_isotropic(
    image: sitk.Image,
    target_spacing: Tuple[float, float, float] = (1.0, 1.0, 1.0),
    default_value: float = -1000.0
) -> sitk.Image:
    """Resample 3D SimpleITK image to isotropic (1.0, 1.0, 1.0) mm spacing."""
    orig_spacing = np.array(image.GetSpacing(), dtype=np.float64)
    orig_size = np.array(image.GetSize(), dtype=np.int64)
    target_spacing = np.array(target_spacing, dtype=np.float64)
    new_size = np.round(orig_size * orig_spacing / target_spacing).astype(np.int64)
    resample = sitk.ResampleImageFilter()
    resample.SetInterpolator(sitk.sitkLinear)
    resample.SetOutputSpacing(target_spacing.tolist())
    resample.SetSize(new_size.tolist())
    resample.SetOutputDirection(image.GetDirection())
    resample.SetOutputOrigin(image.GetOrigin())
    resample.SetDefaultPixelValue(default_value)
    resample.SetOutputPixelType(sitk.sitkFloat32)
    return resample.Execute(image)
def apply_lung_window_and_normalize(
    volume_np: np.ndarray,
    min_hu: float = -1000.0,
    max_hu: float = 400.0
) -> np.ndarray:
    """Clip CT intensities to lung window [-1000, 400] HU and normalize to [0, 1]."""
    clipped = np.clip(volume_np, min_hu, max_hu).astype(np.float32)
    normalized = (clipped - min_hu) / (max_hu - min_hu)
    return normalized
def extract_3d_patch(
    volume_np: np.ndarray,
    centroid_zyx: Tuple[int, int, int],
    patch_size: Tuple[int, int, int] = (64, 64, 64),
    pad_value: float = 0.0
) -> Tuple[np.ndarray, bool]:
    """Extract a 3D subvolume patch (64x64x64) with zero-padding at boundaries."""
    cz, cy, cx = centroid_zyx
    pd, ph, pw = patch_size
    vz, vy, vx = volume_np.shape
    half_d, half_h, half_w = pd // 2, ph // 2, pw // 2
    z_min_req, z_max_req = cz - half_d, cz + (pd - half_d)
    y_min_req, y_max_req = cy - half_h, cy + (ph - half_h)
    x_min_req, x_max_req = cx - half_w, cx + (pw - half_w)
    z_min_vol, z_max_vol = max(0, z_min_req), min(vz, z_max_req)
    y_min_vol, y_max_vol = max(0, y_min_req), min(vy, y_max_req)
    x_min_vol, x_max_vol = max(0, x_min_req), min(vx, x_max_req)
    z_min_patch = z_min_vol - z_min_req
    z_max_patch = z_min_patch + (z_max_vol - z_min_vol)
    y_min_patch = y_min_vol - y_min_req
    y_max_patch = y_min_patch + (y_max_vol - y_min_vol)
    x_min_patch = x_min_vol - x_min_req
    x_max_patch = x_min_patch + (x_max_vol - x_min_vol)
    patch = np.full(patch_size, fill_value=pad_value, dtype=np.float32)
    is_padded = (z_min_req < 0 or z_max_req > vz or y_min_req < 0 or y_max_req > vy or x_min_req < 0 or x_max_req > vx)
    if (z_max_vol > z_min_vol) and (y_max_vol > y_min_vol) and (x_max_vol > x_min_vol):
        patch[z_min_patch:z_max_patch, y_min_patch:y_max_patch, x_min_patch:x_max_patch] = \
            volume_np[z_min_vol:z_max_vol, y_min_vol:y_max_vol, x_min_vol:x_max_vol]
    return patch, is_padded
def parse_scan_nodules(scan: pl.Scan, resampled_sitk_img: sitk.Image, original_sitk_img: sitk.Image) -> List[Dict[str, Any]]:
    """Extract nodule clusters, consensus malignancy ratings, and resampled voxel coordinates."""
    nodules = []
    try:
        clusters = scan.cluster_annotations()
    except Exception as e:
        print(f"Annotation clustering error for {scan.patient_id}: {e}")
        return []
    for c_idx, cluster in enumerate(clusters):
        if not cluster:
            continue

        # Consensus Malignancy (1-5 scale)
        malignancies = [ann.malignancy for ann in cluster if ann.malignancy is not None]
        if not malignancies:
            continue
        consensus_mal = float(np.mean(malignancies))

        # Classification label: 0 (Benign), 1 (Malignant), -1 (Indeterminate)
        if consensus_mal < 3.0:
            mal_class = 0
        elif consensus_mal > 3.0:
            mal_class = 1
        else:
            mal_class = -1
        # Centroid coordinate mapping (DICOM voxel -> World -> 1mm Resampled Voxel)
        world_pts = []
        for ann in cluster:
            ann_vox = ann.centroid  # [y, x, z]
            sitk_idx = (float(ann_vox[1]), float(ann_vox[0]), float(ann_vox[2]))
            try:
                world_pt = original_sitk_img.TransformContinuousIndexToPhysicalPoint(sitk_idx)
                world_pts.append(world_pt)
            except Exception:
                pass
        if not world_pts:
            continue

        avg_world_pt = np.mean(world_pts, axis=0)
        res_cont_idx = resampled_sitk_img.TransformPhysicalPointToContinuousIndex(tuple(avg_world_pt))
        res_voxel_zyx = (
            int(np.round(res_cont_idx[2])),
            int(np.round(res_cont_idx[1])),
            int(np.round(res_cont_idx[0]))
        )
        nodule_entry = {
            "nodule_id": f"{scan.patient_id}_nodule_{c_idx:03d}",
            "patient_id": scan.patient_id,
            "num_annotations": len(cluster),
            "consensus_malignancy": consensus_mal,
            "malignancy_class": mal_class,
            "subtlety": float(np.mean([ann.subtlety for ann in cluster if ann.subtlety])),
            "sphericity": float(np.mean([ann.sphericity for ann in cluster if ann.sphericity])),
            "margin": float(np.mean([ann.margin for ann in cluster if ann.margin])),
            "spiculation": float(np.mean([ann.spiculation for ann in cluster if ann.spiculation])),
            "texture": float(np.mean([ann.texture for ann in cluster if ann.texture])),
            "centroid_world_xyz": tuple(avg_world_pt),
            "centroid_voxel_zyx": res_voxel_zyx,
        }
        nodules.append(nodule_entry)
    return nodules
print("✅ Pipeline functions loaded successfully!")


✅ Pipeline functions loaded successfully!


# Cell 5: Visualization test with 3-plane orthogonal cross-sections

In [ ]:
import matplotlib.pyplot as plt
def plot_nodule_orthogonal_views(volume_np, centroid_zyx, patch_np, title="Nodule Visualization"):
    cz, cy, cx = centroid_zyx
    pz, py, px = patch_np.shape[0] // 2, patch_np.shape[1] // 2, patch_np.shape[2] // 2
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(title, fontsize=15, fontweight="bold")
    # Full volume cross-sections
    axes[0, 0].imshow(volume_np[cz, :, :], cmap="bone", origin="lower")
    axes[0, 0].plot(cx, cy, 'r+', markersize=12, markeredgewidth=2)
    axes[0, 0].set_title(f"Axial (Z={cz})")
    axes[0, 1].imshow(volume_np[:, cy, :], cmap="bone", origin="lower")
    axes[0, 1].plot(cx, cz, 'r+', markersize=12, markeredgewidth=2)
    axes[0, 1].set_title(f"Coronal (Y={cy})")
    axes[0, 2].imshow(volume_np[:, :, cx], cmap="bone", origin="lower")
    axes[0, 2].plot(cy, cz, 'r+', markersize=12, markeredgewidth=2)
    axes[0, 2].set_title(f"Sagittal (X={cx})")
    # Cropped 64x64x64 patch cross-sections
    axes[1, 0].imshow(patch_np[pz, :, :], cmap="bone", origin="lower")
    axes[1, 0].set_title("64x64 Patch (Axial)")
    axes[1, 1].imshow(patch_np[:, py, :], cmap="bone", origin="lower")
    axes[1, 1].set_title("64x64 Patch (Coronal)")
    axes[1, 2].imshow(patch_np[:, :, px], cmap="bone", origin="lower")
    axes[1, 2].set_title("64x64 Patch (Sagittal)")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
# Find patient scans
all_patient_dirs = sorted([p for p in RAW_DATA_DIR.iterdir() if p.is_dir() and p.name.startswith("LIDC-IDRI-")])
print(f"Found {len(all_patient_dirs)} patient directories.")
if all_patient_dirs:
    sample_patient = all_patient_dirs[0].name
    print(f"Testing pipeline on: {sample_patient}")

    dicom_dir = find_dicom_series_dir(all_patient_dirs[0])
    if dicom_dir:
        orig_sitk = load_dicom_volume_sitk(dicom_dir)
        res_sitk = resample_volume_isotropic(orig_sitk, target_spacing=(1.0, 1.0, 1.0))
        vol_np = sitk.GetArrayFromImage(res_sitk)
        vol_norm = apply_lung_window_and_normalize(vol_np)
        scan = pl.query(pl.Scan).filter(pl.Scan.patient_id == sample_patient).first()
        if scan:
            nodules = parse_scan_nodules(scan, res_sitk, orig_sitk)
            if nodules:
                sample_nodule = nodules[0]
                patch, is_pad = extract_3d_patch(vol_norm, sample_nodule["centroid_voxel_zyx"], patch_size=(64, 64, 64))
                title = f"{sample_nodule['nodule_id']} | Consensus Malignancy: {sample_nodule['consensus_malignancy']:.2f}/5.0"
                plot_nodule_orthogonal_views(vol_norm, sample_nodule["centroid_voxel_zyx"], patch, title=title)

Found 25 patient directories.
Testing pipeline on: LIDC-IDRI-0001
Annotation clustering error for LIDC-IDRI-0001: module 'numpy' has no attribute 'int'.
`np.int` was a deprecated alias for the builtin `int`. To avoid this error in existing code, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations


In [ ]:
import numpy as np

# Fix pylidc compatibility with modern NumPy (NumPy >= 1.24 / 2.0)
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'object'):
    np.object = object

print("✅ NumPy compatibility patch applied!")


✅ NumPy compatibility patch applied!


/tmp/ipykernel_576/1948190740.py:10: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'object'):


# Cell 6: Run full batch preprocessing


In [ ]:
from tqdm.notebook import tqdm
def preprocess_dataset(raw_dir: Path, out_dir: Path, max_scans: Optional[int] = None) -> pd.DataFrame:
    patient_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir() and p.name.startswith("LIDC-IDRI-")])
    if max_scans:
        patient_dirs = patient_dirs[:max_scans]
    all_records = []
    print(f"Starting preprocessing for {len(patient_dirs)} patient scans...")
    for p_dir in tqdm(patient_dirs, desc="Processing Patients"):
        patient_id = p_dir.name
        try:
            dicom_dir = find_dicom_series_dir(p_dir)
            if not dicom_dir:
                continue
            orig_sitk = load_dicom_volume_sitk(dicom_dir)
            res_sitk = resample_volume_isotropic(orig_sitk, target_spacing=(1.0, 1.0, 1.0))
            vol_np = sitk.GetArrayFromImage(res_sitk)
            vol_norm = apply_lung_window_and_normalize(vol_np)
            scan = pl.query(pl.Scan).filter(pl.Scan.patient_id == patient_id).first()
            if not scan:
                continue
            nodules = parse_scan_nodules(scan, res_sitk, orig_sitk)
            for nodule in nodules:
                patch, is_pad = extract_3d_patch(vol_norm, nodule["centroid_voxel_zyx"], patch_size=(64, 64, 64))
                patch_path = out_dir / "patches" / f"{nodule['nodule_id']}.pt"

                # Save PyTorch 4D tensor (1, 64, 64, 64)
                tensor_4d = torch.from_numpy(patch).unsqueeze(0).to(torch.float32)
                torch.save({"tensor": tensor_4d, "metadata": nodule}, patch_path)

                record = {**nodule, "patch_path": str(patch_path), "is_padded": is_pad}
                all_records.append(record)
        except Exception as e:
            print(f"⚠️ Error on {patient_id}: {e}")
    manifest_df = pd.DataFrame(all_records)
    manifest_path = out_dir / "manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)
    print(f"🎉 Preprocessing finished! Saved {len(all_records)} patches.")
    print(f"📄 Manifest saved to: {manifest_path}")
    return manifest_df
# Execute (process all scans, or pass max_scans=10 to do a test run first)
manifest_df = preprocess_dataset(RAW_DATA_DIR, OUTPUT_DIR, max_scans=None)
manifest_df.head()


Starting preprocessing for 25 patient scans...


Processing Patients:   0%|          | 0/25 [00:00<?, ?it/s]

🎉 Preprocessing finished! Saved 85 patches.
📄 Manifest saved to: /content/drive/MyDrive/Lung_Nodule_Project/processed_patches/manifest.csv


,nodule_id,patient_id,num_annotations,consensus_malignancy,malignancy_class,subtlety,sphericity,margin,spiculation,texture,centroid_world_xyz,centroid_voxel_zyx,patch_path,is_padded
0,LIDC-IDRI-0001_nodule_000,LIDC-IDRI-0001,4,4.75,1,5.0,3.75,3.25,4.25,4.75,"(56.17468475854846, 86.20332200475828, -115.86...","(224, 258, 222)",/content/drive/MyDrive/Lung_Nodule_Project/pro...,False
1,LIDC-IDRI-0005_nodule_000,LIDC-IDRI-0005,4,2.75,0,2.5,4.75,4.00,1.00,4.75,"(-79.29401867250569, 94.06011877937404, -146.3...","(194, 264, 84)",/content/drive/MyDrive/Lung_Nodule_Project/pro...,False
2,LIDC-IDRI-0005_nodule_001,LIDC-IDRI-0005,4,2.75,0,3.5,4.50,4.50,1.25,4.75,"(44.26583851563347, 100.2974087794774, -139.73...","(200, 270, 208)",/content/drive/MyDrive/Lung_Nodule_Project/pro...,False
3,LIDC-IDRI-0005_nodule_002,LIDC-IDRI-0005,1,2.00,0,3.0,3.00,5.00,1.00,5.00,"(107.00274486666666, 80.08574920000004, -121.2...","(219, 250, 270)",/content/drive/MyDrive/Lung_Nodule_Project/pro...,False
4,LIDC-IDRI-0007_nodule_000,LIDC-IDRI-0007,4,4.75,1,5.0,4.00,3.00,5.00,4.75,"(-48.30369854756778, 44.81288876997458, -90.92...","(269, 226, 152)",/content/drive/MyDrive/Lung_Nodule_Project/pro...,False


# Cell 7: Verify PyTorch Dataset and DataLoader for Phase 2

In [ ]:
from torch.utils.data import Dataset, DataLoader
class LIDCNoduleDataset(Dataset):
    def __init__(self, manifest_csv: str, exclude_indeterminate: bool = True, transform=None):
        self.df = pd.read_csv(manifest_csv)
        if exclude_indeterminate:
            # Exclude indeterminate nodules (malignancy == 3.0)
            self.df = self.df[self.df["malignancy_class"] != -1].reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        data = torch.load(row["patch_path"], weights_only=False)
        patch_tensor = data["tensor"]  # Shape: (1, 64, 64, 64)
        if self.transform:
            patch_tensor = self.transform(patch_tensor)
        label = torch.tensor(row["malignancy_class"], dtype=torch.long)
        malignancy_score = torch.tensor(row["consensus_malignancy"], dtype=torch.float32)

        attributes = torch.tensor([
            row.get("subtlety", 0.0),
            row.get("sphericity", 0.0),
            row.get("margin", 0.0),
            row.get("spiculation", 0.0),
            row.get("texture", 0.0)
        ], dtype=torch.float32)
        return {
            "image": patch_tensor,
            "label": label,
            "malignancy_score": malignancy_score,
            "attributes": attributes,
            "nodule_id": row["nodule_id"],
        }
# Instantiate dataset and batch loader
manifest_csv_path = OUTPUT_DIR / "manifest.csv"
if manifest_csv_path.exists():
    dataset = LIDCNoduleDataset(str(manifest_csv_path))
    loader = DataLoader(dataset, batch_size=4, shuffle=True)
    batch = next(iter(loader))
    print("✅ PyTorch DataLoader test successful!")
    print(f"Batch Image Shape: {batch['image'].shape}")  # Expecting: [4, 1, 64, 64, 64]
    print(f"Batch Labels: {batch['label']}")             # 0 = Benign, 1 = Malignant
    print(f"Batch Malignancy Scores: {batch['malignancy_score']}")

✅ PyTorch DataLoader test successful!
Batch Image Shape: torch.Size([4, 1, 64, 64, 64])
Batch Labels: tensor([0, 0, 1, 0])
Batch Malignancy Scores: tensor([2.6667, 2.5000, 4.6667, 2.5000])
